In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

In [185]:
path = Path("../data/Automotive")

In [186]:
csv_files = sorted(list(path.rglob("*.csv")))

In [187]:
print(f"found {len(csv_files)} files.")

found 19722 files.


In [188]:
csv_list = []

In [189]:
for f in csv_files:
    if f.stat().st_size == 0:
        continue

    try:
        content = pd.read_csv(f, header=None, names=["uid", "class", "px", "py", "wid", "len"])
        content["frame_id"] = f.stem
        content["sequence"] = f.parent.parent.name
        csv_list.append(content)
    except Exception as e:
        print(f"Skipping corrupt file {f}: {e}")

In [190]:
df = pd.concat(csv_list, ignore_index=True)

In [191]:
df

,uid,class,px,py,wid,len,frame_id,sequence
0,22.0,80.0,-0.722190,9.879378,1.7,0.6,0000000003,2019_04_09_bms1000
1,22.0,80.0,-0.681400,9.837713,1.7,0.6,0000000004,2019_04_09_bms1000
2,22.0,80.0,-0.640854,9.788227,1.7,0.6,0000000005,2019_04_09_bms1000
3,22.0,80.0,-0.844815,9.600132,1.7,0.6,0000000006,2019_04_09_bms1000
4,22.0,80.0,-0.604718,9.764450,1.7,0.6,0000000007,2019_04_09_bms1000
...,...,...,...,...,...,...,...,...
38220,7.0,80.0,11.682308,0.914115,0.6,1.7,0000000202,2019_05_29_pcms005
38221,7.0,80.0,11.682308,0.914115,0.6,1.7,0000000203,2019_05_29_pcms005
38222,7.0,80.0,11.682308,0.914115,0.6,1.7,0000000204,2019_05_29_pcms005
38223,7.0,80.0,11.682308,0.914115,0.6,1.7,0000000205,2019_05_29_pcms005


In [192]:
df.dtypes

uid         float64
class       float64
px          float64
py          float64
wid         float64
len         float64
frame_id        str
sequence        str
dtype: object

In [193]:
df["uid"] = pd.to_numeric(df["uid"], errors="coerce").fillna(-1).astype(int)
df["class"] = (pd.to_numeric(df["class"], errors="coerce").fillna(-1).astype(int))
for col in ["px", "py", "wid", "len"]:
    df[col] = pd.to_numeric(df[col], errors="coerce").astype(float)

In [194]:
df["range_m"] = np.sqrt(df["px"] ** 2 + df["py"] ** 2)

In [195]:
df["azimuth_deg"] = np.degrees(np.arctan2(df["px"], df["py"]))

In [196]:
label_map = {0: 'person',
            2: 'car',
            3: 'motorbike',
            5: 'bus',
            7: 'truck',
            80: 'cyclist'
            }

df["class_name"] = df["class"].map(label_map).fillna('other')

In [197]:
df

,uid,class,px,py,wid,len,frame_id,sequence,range_m,azimuth_deg,class_name
0,22,80,-0.722190,9.879378,1.7,0.6,0000000003,2019_04_09_bms1000,9.905740,-4.180929,cyclist
1,22,80,-0.681400,9.837713,1.7,0.6,0000000004,2019_04_09_bms1000,9.861283,-3.962209,cyclist
2,22,80,-0.640854,9.788227,1.7,0.6,0000000005,2019_04_09_bms1000,9.809183,-3.745918,cyclist
3,22,80,-0.844815,9.600132,1.7,0.6,0000000006,2019_04_09_bms1000,9.637232,-5.029091,cyclist
4,22,80,-0.604718,9.764450,1.7,0.6,0000000007,2019_04_09_bms1000,9.783158,-3.543832,cyclist
...,...,...,...,...,...,...,...,...,...,...,...
38220,7,80,11.682308,0.914115,0.6,1.7,0000000202,2019_05_29_pcms005,11.718017,85.525848,cyclist
38221,7,80,11.682308,0.914115,0.6,1.7,0000000203,2019_05_29_pcms005,11.718017,85.525848,cyclist
38222,7,80,11.682308,0.914115,0.6,1.7,0000000204,2019_05_29_pcms005,11.718017,85.525848,cyclist
38223,7,80,11.682308,0.914115,0.6,1.7,0000000205,2019_05_29_pcms005,11.718017,85.525848,cyclist


In [200]:
print(df["class_name"].value_counts(dropna=False))
print("\nNull count per column:\n", df.isna().sum())

class_name
person       14849
car          12369
cyclist      10253
truck          454
motorbike      194
other          106
Name: count, dtype: int64

Null count per column:
 uid            0
class          0
px             0
py             0
wid            0
len            0
frame_id       0
sequence       0
range_m        0
azimuth_deg    0
class_name     0
dtype: int64


In [198]:
import pyarrow as pa
import pyarrow.parquet as pq

In [199]:
table = pa.Table.from_pandas(df)
pq.write_table(table, "../data/metadata.parquet")

checking if the file was uploaded correctly

In [2]:
df_remote = pd.read_parquet(
    "hf://datasets/hany34/raw-adc-data-77ghz-mmwave-radar-automotive-object-detection/metadata.parquet"
)
print("Uploaded successfully! Remote shape:", df_remote.shape)

d:\github\radar-dsp-core\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Uploaded successfully! Remote shape: (38225, 11)
